# Prática 17/11 · Árvores de decisão e sobreajuste: votos brancos e nulos em MT
**Aprendizagem Computacional · 4º TIA · Aula 04 · versão INICIAL**

**Pergunta:** o perfil do eleitorado ajuda a prever o **percentual de votos brancos e nulos** de cada município de MT (1º turno para presidente, 2026)?
Uma Justiça Eleitoral que quisesse reforçar a orientação ao eleitor poderia usar algo assim para priorizar municípios. Aqui é um exercício: o objetivo é ver, com números, o que acontece quando um modelo fica **complexo demais** (cap. 10 e 16).

| Parte | O que você faz | Apostila |
|---|---|---|
| 1–2 | Carregar e olhar o novo alvo (pronto) | cap. 4 |
| 3 | **TODO 1:** X, y e separação | cap. 9, 11 |
| 4 | **TODO 2:** baseline e regressão linear na validação cruzada | cap. 9, 11 |
| 5 | **TODO 3:** a curva de sobreajuste da árvore | cap. 10, 16 |
| 6 | Desenhar e ler a árvore (pronto) | cap. 16 |
| 7 | **TODO 4:** Random Forest e importância das variáveis | cap. 18 |
| 8 | **TODO 5:** escolher e abrir o teste UMA vez | cap. 9 |
| 9 | Verificação final | — |

Não precisa instalar nada no Colab. O notebook não grava nenhum arquivo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42
FEATURES = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO']
pd.set_option('display.precision', 3)

## 1 · Carregar os dados (pronto)
142 municípios de MT. Além do perfil do eleitorado, a base traz `QT_COMPARECIMENTO`, `QT_VOTOS_BRANCOS` e `QT_TOTAL_VOTOS_NULOS` (TSE, resultado por município e zona, somado por município) e a coluna calculada:

`PCT_BRANCOS_NULOS = 100 × (brancos + nulos) / comparecimento`

In [ ]:
ARQUIVO = 'municipios_mt_votos.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-11-17_Aula04_Arvores_Sobreajuste/dados/municipios_mt_votos.csv'

candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
if local is not None:
    municipios = pd.read_csv(local)
    print('Lido do arquivo:', local.resolve())
else:
    municipios = pd.read_csv(URL)
    print('Lido do GitHub da disciplina')
print(municipios.shape)

## 2 · Olhar o alvo (pronto)

In [ ]:
print(municipios['PCT_BRANCOS_NULOS'].describe())
print('\nCorrelação de cada feature com o alvo:')
print(municipios[FEATURES + ['PCT_BRANCOS_NULOS']].corr()['PCT_BRANCOS_NULOS'].drop('PCT_BRANCOS_NULOS').round(2))

municipios['PCT_BRANCOS_NULOS'].plot.hist(bins=20, figsize=(6, 3), title='Votos brancos e nulos (% do comparecimento)')
plt.show()

Os valores são baixos (em torno de 3%) e variam pouco entre municípios. A feature mais ligada ao alvo é `PCT_65_MAIS`: municípios com eleitorado mais velho têm mais brancos e nulos.

## 3 · TODO 1: X, y e separação (cap. 9 e 11)
- `y` = coluna `PCT_BRANCOS_NULOS` (agora é um **número**: regressão, como na Aula08 e no PPI-molde);
- `X` = colunas de `FEATURES`;
- `train_test_split(X, y, test_size=0.25, random_state=SEMENTE)`. Em regressão não usamos `stratify`.

Não entram no X: `QT_VOTOS_BRANCOS` e `QT_TOTAL_VOTOS_NULOS` (são a resposta) nem os identificadores.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO 1
y = None
X = None
X_treino, X_teste, y_treino, y_teste = None, None, None, None

print('treino:', X_treino.shape, '| teste:', X_teste.shape)

## 4 · TODO 2: as réguas, baseline e regressão linear (cap. 9 e 11)
A função `mae_cv` (pronta) roda a validação cruzada com 5 folds e devolve dois números:
- **MAE no treino:** o erro médio nos dados que o modelo **viu** (média dos 5 folds);
- **MAE na validação:** o erro médio nos dados que ele **não viu**. É este que importa.

MAE (erro absoluto médio) está na unidade do alvo: um MAE de 0,40 é errar, em média, 0,40 ponto percentual.

Complete os dois modelos: `DummyRegressor()` (prevê sempre a média do treino) e `make_pipeline(StandardScaler(), LinearRegression())`.

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)


def mae_cv(modelo):
    r = cross_validate(modelo, X_treino, y_treino, cv=cv, scoring='neg_mean_absolute_error', return_train_score=True)
    return abs(r['train_score'].mean()), abs(r['test_score'].mean())   # o sklearn devolve o MAE com sinal negativo


# TODO 2
baseline = None
linear = None

resultados = {'Baseline (média)': mae_cv(baseline), 'Regressão linear': mae_cv(linear)}
for nome, (treino, valid) in resultados.items():
    print(f'{nome:<18} MAE treino {treino:.3f} | MAE validação {valid:.3f}')

## 5 · TODO 3: a curva de sobreajuste (cap. 10 e 16)
Uma **árvore de decisão** faz perguntas do tipo "`PCT_65_MAIS` ≤ 12,2?" e vai dividindo os municípios em grupos cada vez menores. A **profundidade** é quantas perguntas seguidas ela pode fazer.
Dentro do `for`, crie `DecisionTreeRegressor(max_depth=prof, random_state=SEMENTE)` e chame `mae_cv` nele. `None` quer dizer "sem limite".

In [ ]:
from sklearn.tree import DecisionTreeRegressor

PROFUNDIDADES = [1, 2, 3, 4, 5, 6, 8, 10, None]
curva = []
for prof in PROFUNDIDADES:
    # TODO 3: arvore = ...; treino, valid = mae_cv(arvore)
    arvore = None
    treino, valid = None, None
    curva.append({'profundidade': str(prof), 'MAE treino': treino, 'MAE validação': valid})

curva = pd.DataFrame(curva)
print(curva.round(3).to_string(index=False))

ax = curva.plot(x='profundidade', y=['MAE treino', 'MAE validação'], marker='o', figsize=(7, 3.5),
                title='Árvore: erro no treino × erro na validação')
ax.axhline(resultados['Baseline (média)'][1], color='grey', ls='--', label='baseline (validação)')
ax.set_ylabel('MAE (p.p.)'); ax.legend(); plt.show()

**Pergunta para discussão:** sem limite de profundidade, o erro no treino vai a zero. Por que o erro na validação piora?

**Resposta esperada:** com perguntas suficientes, a árvore isola cada município numa folha e **decora** o treino, inclusive o ruído (variações que não se repetem). Num município novo, essas regras decoradas não valem. Isso é **sobreajuste** (cap. 10): complexidade demais para a quantidade de dados. Com 106 municípios e pouco sinal, a árvore de 1 pergunta já é a melhor, e as mais fundas ficam até piores que o baseline.

Analogia: ajustar uma equação de volume com 10 coeficientes a 8 árvores cubadas. Ela passa exatamente por todas, e erra feio na nona.

## 6 · Desenhar e ler a árvore (pronto, cap. 16)
Uma árvore pequena é um modelo que dá para **ler**: cada caminho é uma regra "se… então…".

In [ ]:
from sklearn.tree import plot_tree, export_text

arvore_2 = DecisionTreeRegressor(max_depth=2, random_state=SEMENTE).fit(X_treino, y_treino)
print(export_text(arvore_2, feature_names=FEATURES, decimals=2))
plt.figure(figsize=(11, 4.5))
plot_tree(arvore_2, feature_names=FEATURES, filled=True, rounded=True, precision=2, fontsize=9)
plt.show()

Leitura: a primeira pergunta usa `PCT_65_MAIS`, a feature mais correlacionada com o alvo. O valor em cada folha é a **média** dos municípios de treino que caíram ali: é isso que a árvore prevê para um município novo que siga aquele caminho.

## 7 · TODO 4: Random Forest e importância das variáveis (cap. 18)
Crie `RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=SEMENTE)`, avalie com `mae_cv` e depois ajuste no treino (`fit`) para ver as importâncias.
`min_samples_leaf=3` proíbe folhas com menos de 3 municípios: é um freio contra decorar.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO 4
floresta = None
resultados['Random Forest'] = mae_cv(floresta)
floresta.fit(X_treino, y_treino)

importancias = pd.Series(floresta.feature_importances_, index=FEATURES).sort_values()
print(importancias.round(2))
importancias.plot.barh(figsize=(6, 2.8), title='Importância das variáveis (Random Forest)'); plt.show()

**Cuidado ao ler importância:** ela diz quanto cada coluna foi usada para reduzir o erro **neste modelo**, e não que a coluna **causa** o alvo. E colunas parecidas entre si dividem a importância.

## 8 · TODO 5: escolher e abrir o teste UMA vez (cap. 9)
A tabela abaixo (pronta) junta todos os candidatos pelo **MAE na validação**. Escolha o menor. Pelos números, é a **árvore de profundidade 1**.
1. crie `escolhido = DecisionTreeRegressor(max_depth=1, random_state=SEMENTE)` e ajuste no treino;
2. preveja `X_teste` e guarde em `prev_teste`.

In [ ]:
resultados['Árvore (prof. 1)'] = tuple(curva.loc[0, ['MAE treino', 'MAE validação']])
resultados['Árvore (sem limite)'] = tuple(curva.loc[len(curva) - 1, ['MAE treino', 'MAE validação']])
tabela = pd.DataFrame(resultados, index=['MAE treino', 'MAE validação']).T.sort_values('MAE validação')
print(tabela.round(3))

In [ ]:
from sklearn.metrics import mean_absolute_error

# TODO 5
escolhido = None
prev_teste = None

mae_teste = mean_absolute_error(y_teste, prev_teste)
mae_baseline_teste = mean_absolute_error(y_teste, np.full(len(y_teste), y_treino.mean()))
print(f'TESTE · árvore (prof. 1): MAE {mae_teste:.3f} | baseline: MAE {mae_baseline_teste:.3f}')

**Pergunta para discussão:** a diferença entre a árvore de 1 pergunta, a regressão linear e o Random Forest na validação é de poucos centésimos. Qual você colocaria em produção, e por quê?

**Resposta esperada:** com diferenças tão pequenas (menores que a oscilação entre folds), vale o critério de simplicidade: a árvore de 1 pergunta, ou a linear, é fácil de explicar ao gestor ("onde o eleitorado tem mais de 12% de idosos, espere mais brancos e nulos"). O Random Forest custa mais e não ganhou. Mais complexo não é sinônimo de melhor.

## 9 · Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('treino com 106 e teste com 36 municípios', lambda: len(X_treino) == 106 and len(X_teste) == 36 and y.name == 'PCT_BRANCOS_NULOS'),
    conferir('baseline: MAE na validação 0,417', lambda: round(resultados['Baseline (média)'][1], 3) == 0.417),
    conferir('linear: MAE na validação 0,379', lambda: round(resultados['Regressão linear'][1], 3) == 0.379),
    conferir('árvore sem limite: MAE no treino 0,000 e na validação 0,511', lambda: abs(round(curva['MAE treino'].iloc[-1], 3)) == 0 and round(curva['MAE validação'].iloc[-1], 3) == 0.511),
    conferir('Random Forest: MAE na validação 0,391', lambda: round(resultados['Random Forest'][1], 3) == 0.391),
    conferir('teste: a árvore de prof. 1 erra menos que o baseline', lambda: mae_teste < mae_baseline_teste),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist (as 6 perguntas da PPI)
1. **y?** % de brancos e nulos (número). 2. **Baseline?** a média do treino. 3. **Métrica?** MAE, em pontos percentuais, fácil de explicar.
4. **Split?** teste separado no início; profundidade escolhida pela CV. 5. **Vazamento?** as contagens de brancos e nulos ficaram fora do X.
6. **Evidência?** o ganho sobre o baseline é pequeno (cerca de 0,05 p.p. na validação). Diga isso no artigo, sem vender o modelo.

**Para quem terminar:** use `GridSearchCV` na árvore com `{'max_depth': [1, 2, 3, 4], 'min_samples_leaf': [1, 5, 10, 20]}` e `scoring='neg_mean_absolute_error'`. Folhas maiores ajudam a árvore mais funda?